# omnomonomononon - uniform

In [12]:
import igraph as ig
import numpy as np
import warnings

In [13]:
pip install infomap

Note: you may need to restart the kernel to use updated packages.


In [14]:
# apparently i need this
def safe_xlogx(x):
    return np.where(x > 0, x * np.log2(x), 0.0)

In [15]:
def pagerank_nonuniform(M, tau=0.15, tol=1e-15, maxiter=int(1e6)):
    N = M.shape[0]
    row_sums = M.sum(axis=1)          # out-strength
    dangling = (row_sums == 0)
    row_sums_safe = np.where(dangling, 1, row_sums)
    M_norm = M / row_sums_safe[:, None]   # row-stochastic

    # teleportation proportional to out-strength (Eq. 4 in tutorial)
    total_out = row_sums.sum()
    d = row_sums / total_out if total_out > 0 else np.ones(N) / N

    # STEP 1: solve for p* (intermediate distribution)
    p_star = np.ones(N) / N
    for _ in range(maxiter):
        dangling_sum = p_star[dangling].sum()
        p_star_new = (1 - tau) * (p_star @ M_norm + dangling_sum * d) + tau * d
        if np.linalg.norm(p_star_new - p_star) < tol:
            p_star = p_star_new
            break
        p_star = p_star_new

    # STEP 2: one extra link-only step to get recorded visit rates (Eq. 6)
    p = p_star @ M_norm
    return p

In [16]:
# exit flow for directed: proportional to p[i] * w_ij / out_strength[i]
def compute_exit_flow_nonuniform(g, communities, p):
    communities = np.array(communities)
    out_strength = np.array(g.strength(mode="out", weights="weight" if g.is_weighted() else None), dtype=float)
    weights = np.array(g.es["weight"] if g.is_weighted() else [1.0] * g.ecount())
    edges = np.array(g.get_edgelist(), dtype=int)

    src, trg = edges[:, 0], edges[:, 1]
    src_com, trg_com = communities[src], communities[trg]
    betw = src_com != trg_com

    # avoid div by zero
    out_str_safe = np.where(out_strength > 0, out_strength, 1.0)
    flow = p[src] * weights / out_str_safe[src]

    exit_flow = np.zeros(max(communities) + 1)
    np.add.at(exit_flow, src_com[betw], flow[betw])
    return exit_flow

In [17]:
# exit weights for undirected
def compute_exit_weights(g, communities):
    communities = np.array(communities)
    weights = np.array(g.es["weight"] if g.is_weighted() else [1.0] * g.ecount())
    edges = np.array(g.get_edgelist(), dtype=int)

    src_com = communities[edges[:, 0]]
    trg_com = communities[edges[:, 1]]
    betw = src_com != trg_com

    exit_weights = np.zeros(max(communities) + 1)
    np.add.at(exit_weights, src_com[betw], weights[betw])
    np.add.at(exit_weights, trg_com[betw], weights[betw])
    return exit_weights

In [18]:
# map equation for (un)directed + (un)weighted with non-uniform teleportation 
def compute_description_length(g, communities, tau=0.15):
    communities = np.array(communities)
    num_comms = max(communities) + 1
    N = g.vcount()

    if g.is_directed():
        adj = np.array(g.get_adjacency(attribute="weight" if g.is_weighted() else None).data, dtype=float)
        p = pagerank_nonuniform(adj, tau=tau)
        
        p_mod = np.zeros(num_comms)
        np.add.at(p_mod, communities, p)
        
        exit_flow = compute_exit_flow_nonuniform(g, communities, p)
        
        # unrecorded teleportation: jumps never encoded, so only edge exits count
        q_mod = exit_flow

    else:
        weights = np.array(g.es["weight"] if g.is_weighted() else [1.0] * g.ecount())
        total_weight_x2 = 2 * np.sum(weights)

        strength = np.array(g.strength(weights="weight" if g.is_weighted() else None), dtype=float)
        p = strength / total_weight_x2

        p_mod = np.zeros(num_comms)
        np.add.at(p_mod, communities, p)

        exit_weights = compute_exit_weights(g, communities)
        q_mod = exit_weights / total_weight_x2

    q_sum = np.sum(q_mod)
    p_loop = p_mod + q_mod

    L = safe_xlogx(q_sum) - 2 * np.sum(safe_xlogx(q_mod)) \
        - np.sum(safe_xlogx(p)) + np.sum(safe_xlogx(p_loop))

    return L

# testing on Laura's igraphs

In [21]:
# unweighted + undirected
g = ig.Graph.Read_GraphML("C:/Users/savin/ITI/unweighted_undirected.graphml")
communities = g.community_infomap()
L_custom = compute_description_length(g, communities.membership)
L_igraph = communities.codelength
print("--- unweighted + undirected ---")
print("Custom L:", L_custom)
print("igraph L:", L_igraph)

# weighted + undirected
g = ig.Graph.Read_GraphML("C:/Users/savin/ITI/weighted_undirected.graphml")
communities = g.community_infomap(edge_weights="weight")
L_custom = compute_description_length(g, communities.membership)
L_igraph = communities.codelength
print("\n--- weighted + undirected ---")
print("Custom L:", L_custom)
print("igraph L:", L_igraph)

# unweighted + directed
g = ig.Graph.Read_GraphML("C:/Users/savin/ITI/unweighted_directed.graphml")
communities = g.community_infomap()
L_custom = compute_description_length(g, communities.membership)
L_igraph = communities.codelength
print("\n--- unweighted + directed ---")
print("Custom L:", L_custom)
print("igraph L:", L_igraph)

# weighted + directed
g = ig.Graph.Read_GraphML("C:/Users/savin/ITI/weighted_directed.graphml")
communities = g.community_infomap(edge_weights="weight")
L_custom = compute_description_length(g, communities.membership)
L_igraph = communities.codelength
print("\n--- weighted + directed ---")
print("Custom L:", L_custom)
print("igraph L:", L_igraph)

--- unweighted + undirected ---
Custom L: 3.4015829735877707
igraph L: 3.40158297358777

--- weighted + undirected ---
Custom L: 3.3367124727600905
igraph L: 3.336712472760091

--- unweighted + directed ---
Custom L: 3.0758500813869554
igraph L: 3.522512525113249

--- weighted + directed ---
Custom L: 2.6423322218077088
igraph L: 3.3964670718825403


C:\Users\savin\AppData\Local\Temp\ipykernel_7692\3689050749.py:3: RuntimeWarning: divide by zero encountered in log2
  return np.where(x > 0, x * np.log2(x), 0.0)
C:\Users\savin\AppData\Local\Temp\ipykernel_7692\3689050749.py:3: RuntimeWarning: invalid value encountered in multiply
  return np.where(x > 0, x * np.log2(x), 0.0)


# wtf is happening 

In [25]:
from infomap import Infomap

def run_official_infomap(g, weighted=False):
    im = Infomap("--directed --two-level --silent")
    for e in g.es:
        w = e["weight"] if weighted else 1.0
        im.add_link(e.source, e.target, w)
    im.run()
    return im.codelength

def pct_diff(value, reference):
    return 100 * (value - reference) / reference

def compare(g, weighted, label):
    communities = g.community_infomap(edge_weights="weight" if weighted else None)
    L_custom = compute_description_length(g, communities.membership)
    L_igraph = communities.codelength
    L_official = run_official_infomap(g, weighted=weighted)
    
    print(f"--- {label} ---")
    print(f"Custom L:   {L_custom:.6f}   ({pct_diff(L_custom, L_official):+.2f}% vs official)")
    print(f"igraph L:   {L_igraph:.6f}   ({pct_diff(L_igraph, L_official):+.2f}% vs official)")
    print(f"Official L: {L_official:.6f}   (official)\n")

g = ig.Graph.Read_GraphML("C:/Users/savin/ITI/unweighted_directed.graphml")
compare(g, weighted=False, label="unweighted + directed")

g = ig.Graph.Read_GraphML("C:/Users/savin/ITI/weighted_directed.graphml")
compare(g, weighted=True, label="weighted + directed")

--- unweighted + directed ---
Custom L:   3.075850   (-1.63% vs official)
igraph L:   3.522513   (+12.66% vs official)
Official L: 3.126697   (official)

--- weighted + directed ---
Custom L:   2.622879   (-3.32% vs official)
igraph L:   3.392047   (+25.03% vs official)
Official L: 2.713013   (official)



C:\Users\savin\AppData\Local\Temp\ipykernel_7692\3689050749.py:3: RuntimeWarning: divide by zero encountered in log2
  return np.where(x > 0, x * np.log2(x), 0.0)
C:\Users\savin\AppData\Local\Temp\ipykernel_7692\3689050749.py:3: RuntimeWarning: invalid value encountered in multiply
  return np.where(x > 0, x * np.log2(x), 0.0)
